# Statistical Helper Functions

This notebook contains functions for statistically screening features before including them in the model. We use chi-square tests for binary/categorical features and point-biserial correlation for numeric features.

The goal is to filter out features that either don't correlate with churn or have too much missing data.

## Imports

Bringing in scipy for statistical tests and pandas for data handling.

In [0]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency, pointbiserialr

## Screening Rules

We'll drop features if they meet any of these criteria:
- **High nulls:** More than 50% missing values
- **Not significant:** p-value >= 0.05 (not statistically significant)
- **Wrong direction:** Negative correlation with churn (doesn't make sense)
- **Leakage risk:** Too perfect correlation (likely data leakage)

In [0]:
# Thresholds for feature screening
NULL_THRESHOLD = 50.0  # Drop if more than 50% null
P_THRESHOLD = 0.05     # Drop if p-value >= 0.05

## Binary Feature Screening

This function tests binary or categorical features using chi-square test. It checks if the feature has a statistically significant relationship with the target variable (churn).

In [0]:
def screen_binary_col(df, col, target_col='Target', yes_val='Yes'):
    """
    Screen a binary/categorical column using chi-square test.
    
    Args:
        df: pandas DataFrame
        col: Column name to screen
        target_col: Target variable column (default 'Target')
        yes_val: Value to treat as positive class (default 'Yes')
    
    Returns:
        dict with screening results (null_pct, p_value, decision, reason)
    """
    # Calculate null percentage
    null_pct = df[col].isna().mean() * 100
    
    # Filter out nulls for testing
    sub = df[[col, target_col]].dropna()
    sub_bin = (sub[col] == yes_val).astype(int)
    
    # Check if column has variation
    if sub_bin.nunique() < 2:
        return {
            'col': col,
            'null_pct': null_pct,
            'p_value': None,
            'decision': 'DROP',
            'reason': 'No variation (all same value)'
        }
    
    # Apply chi-square test
    try:
        contingency = pd.crosstab(sub_bin, sub[target_col])
        chi2, p_val, _, _ = chi2_contingency(contingency)
        
        # Determine if we should keep or drop
        if null_pct >= NULL_THRESHOLD:
            decision = 'DROP'
            reason = f'High nulls ({null_pct:.1f}%)'
        elif p_val >= P_THRESHOLD:
            decision = 'DROP'
            reason = f'Not significant (p={p_val:.3f})'
        else:
            decision = 'KEEP'
            reason = f'Significant (p={p_val:.3f})'
        
        return {
            'col': col,
            'null_pct': null_pct,
            'p_value': p_val,
            'decision': decision,
            'reason': reason
        }
    except Exception as e:
        return {
            'col': col,
            'null_pct': null_pct,
            'p_value': None,
            'decision': 'DROP',
            'reason': f'Test failed: {str(e)}'
        }

## Numeric Feature Screening

This function tests numeric features using point-biserial correlation. It measures the correlation between a continuous variable and a binary target.

We also check for negative correlations (wrong direction) since most features should increase with churn risk.

In [0]:
def screen_numeric_col(df, col, target_col='Target'):
    """
    Screen a numeric column using point-biserial correlation.
    
    Args:
        df: pandas DataFrame
        col: Column name to screen
        target_col: Target variable column (default 'Target')
    
    Returns:
        dict with screening results (null_pct, correlation, p_value, decision, reason)
    """
    # Calculate null percentage
    null_pct = df[col].isna().mean() * 100
    
    # Convert to numeric and filter nulls
    sub = df[[col, target_col]].copy()
    sub[col] = pd.to_numeric(sub[col], errors='coerce')
    sub = sub.dropna()
    
    # Check if column has variation
    if sub[col].nunique() < 2:
        return {
            'col': col,
            'null_pct': null_pct,
            'correlation': None,
            'p_value': None,
            'decision': 'DROP',
            'reason': 'No variation'
        }
    
    # Apply point-biserial correlation
    try:
        corr, p_val = pointbiserialr(sub[target_col], sub[col])
        
        # Determine if we should keep or drop
        if null_pct >= NULL_THRESHOLD:
            decision = 'DROP'
            reason = f'High nulls ({null_pct:.1f}%)'
        elif p_val >= P_THRESHOLD:
            decision = 'DROP'
            reason = f'Not significant (p={p_val:.3f})'
        elif corr < 0:
            decision = 'DROP'
            reason = f'Negative correlation (r={corr:.3f})'
        else:
            decision = 'KEEP'
            reason = f'Significant (r={corr:.3f}, p={p_val:.3f})'
        
        return {
            'col': col,
            'null_pct': null_pct,
            'correlation': corr,
            'p_value': p_val,
            'decision': decision,
            'reason': reason
        }
    except Exception as e:
        return {
            'col': col,
            'null_pct': null_pct,
            'correlation': None,
            'p_value': None,
            'decision': 'DROP',
            'reason': f'Test failed: {str(e)}'
        }

## Confirmation

Confirming the statistical functions are loaded.

In [0]:
print("✓ Statistical helper functions loaded")
print("  Available: screen_binary_col(), screen_numeric_col()")
print(f"  Thresholds: null>={NULL_THRESHOLD}%, p>={P_THRESHOLD}")